In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

INPUT="results/summary.csv"

OUTDIR="analysis"

os.makedirs(OUTDIR,exist_ok=True)

df=pd.read_csv(INPUT)

OUTLIERS=["ZScore","IsolationForest","IQR"]

models=sorted(df["model"].unique())

rows=[]

for model in models:

    mdf=df[df["model"]==model]

    datasets=mdf["dataset"].unique()

    for d in datasets:

        sub=mdf[mdf["dataset"]==d]

        base=sub[sub["outlier"].isna()]

        if len(base)==0:
            continue

        base_mean=base.iloc[0]["mean"]

        for o in OUTLIERS:

            filt=sub[sub["outlier"]==o]

            if len(filt)==0:
                continue

            r=filt.iloc[0]

            delta=r["mean"]-base_mean

            removed=r["removed_pct"]*100

            efficiency=0

            if removed>0:

                efficiency=delta/removed

            rows.append({

                "dataset":d,

                "model":model,

                "method":o,

                "mean":r["mean"],

                "std":r["std"],

                "baseline":base_mean,

                "delta":delta,

                "abs_delta":abs(delta),

                "removed_pct":removed,

                "efficiency":efficiency,

                "improved":delta>0

            })

enriched=pd.DataFrame(rows)

# average removal per model/method

avg_removed=enriched.groupby(

["model","method"]

)["removed_pct"].mean().reset_index()

avg_removed=avg_removed.rename(

columns={"removed_pct":"avg_removed_pct"}

)

enriched=enriched.merge(

avg_removed,

on=["model","method"]

)

enriched.to_csv(

f"{OUTDIR}/enriched_results.csv",

index=False
)

print("CSV with derived metrics created")


###################################
# SCATTER PLOTS
###################################

colors={

"ZScore":"blue",

"IsolationForest":"green",

"IQR":"red"

}

for model in models:

    sub=enriched[enriched["model"]==model]

    plt.figure()

    for method in OUTLIERS:

        m=sub[sub["method"]==method]

        plt.scatter(

            m["removed_pct"],

            m["delta"],

            label=method,

            alpha=0.7
        )

    plt.axhline(0)

    plt.xlabel("Removed %")

    plt.ylabel("Delta performance")

    plt.title(f"{model} : Outlier removal impact")

    plt.legend()

    plt.savefig(

        f"{OUTDIR}/{model}_scatter.png",

        dpi=150,

        bbox_inches="tight"
    )

    plt.close()

print("Scatter plots created")

CSV with derived metrics created
Scatter plots created


In [2]:
!pip install matplotlib

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai